# Mapeamento Eleitoral e Visualizacao Geospacial com Folium

Trabalho feito por: Joao Pedro, Rian e Rodrigo

O trabalho exibe o mapeamento eleitoral na cidade de Divinópolis para os candidatos a presidência no 1° turno e no 2° turno e utiliza o Folium para a visualização geoespacial do mapa.

# Organização dos Dados e Filtragem

Nas células abaixo é exibido a forma que foi realizada a organização e filtragem dos dados, começando pela leitura do CSV que foi fornecido pelo TSE que informa os votos para presidente por zona e seção de todos os municípios do país, com o próximo passo tomado sendo o de filtrar esses dados para corresponderem apenas aos das zonas e seções que correspondem ao município de Divinópolis e separá-los entre os do 1° e 2° turno das eleições juntamente da contagem de votos para cada candidato em cada uma dessas seções.

Além da organização com o CSV fornecido pelo TSE, após juntar os endereços e nomes dos locais das seções em um novo CSV foi possível utilizar um script para encontrar as coordenadas dos endereços que seriam utilizados no mapa criado pelo folium, com o script localizando o endereço e realizando filtragens básicas de detalhes que podem atrapalhar durante a busca dos locais na API.

Durante o uso do script para localizar as coordenadas dos endereços, houve uma dificuldade de encontrar as localidades em zonas rurais da cidade, com grande parte dessas coordenadas sendo preenchidas manualmente ou em localidades aproximadas em distritos e zonas afastadas. 
Também houveram problemas com "sujeiras" em endereços localizados dentro da cidade, com esses erros vindos do próprio endereço fornecido pelo CSV do TSE que acabou ficando com detalhes que impediram a localização do local da seção pela API, como por exemplo fornecendo nomes antigos do local (como referência) na parte de endereço, siglas ou adições ao nome do local que passaram pelo filtro definido, inexistência da marcação do local em mapas, ou simplesmente endereços incompletos ou incorretos.

Após o uso do script e preenchimento manual de certas coordenadas, foi realizada a geolocalização reversa dessas coordenadas para definir bairros e referências de onde as seções de votação estão localizadas.


In [1]:
# Lendo o CSV com os dados de votação informados pelo TSE 
import pandas as pd

df = pd.read_csv("votacao_secao_2026_BR.csv", sep=';', encoding='latin-1')

In [4]:
# Filtragem dos dados para o município de Divinópolis e transformação do dataframe dos votos locais em CSV

colunas_interesse = ['ANO_ELEICAO', 'NR_TURNO', 'NR_ZONA', 'NR_SECAO', 'NM_MUNICIPIO', 'NM_VOTAVEL', 'DS_CARGO', 'QT_VOTOS', 'NM_LOCAL_VOTACAO', 'DS_LOCAL_VOTACAO_ENDERECO', 'NR_LOCAL_VOTACAO']
df_divinopolis = df[df["NM_MUNICIPIO"] == "DIVINÓPOLIS"][colunas_interesse]

# Filtragem dos dados para os candidatos a presidente e soma dos votos por candidato em cada zona/seção eleitoral da cidade no 1° turno
df_presidente_grouped_1turno = df_divinopolis[df_divinopolis['NR_TURNO'] == 1].groupby(['NR_ZONA', 'NR_SECAO', 'NR_TURNO', 'NM_VOTAVEL'])['QT_VOTOS'].sum().reset_index()

# Filtragem dos dados para os candidatos a presidente e soma dos votos por candidato em cada zona/seção eleitoral da cidade no 2° turno
df_presidente_grouped_2turno = df_divinopolis[df_divinopolis['NR_TURNO'] == 2].groupby(['NR_ZONA', 'NR_SECAO', 'NR_TURNO', 'NM_VOTAVEL'])['QT_VOTOS'].sum().reset_index()
print(df_presidente_grouped_2turno)

# Totais de votos dos candidatos no 1° turno
totais_candidatos = df_presidente_grouped_1turno.groupby('NM_VOTAVEL')['QT_VOTOS'].sum().sort_values(ascending=False)
principais_candidatos = totais_candidatos.head(5).index.tolist()
print('Candidatos mapeados 1° turno (5 maiores totais de votos válidos):')
display(totais_candidatos.head(5).rename('Votos').to_frame())

# Criação CSV com os dados de votação do município de Divinópolis
df_divinopolis = df_divinopolis.drop_duplicates()
df_divinopolis.to_csv('votacao_secao_2026_Divinopolis.csv', sep=';', index=False)

      NR_ZONA  NR_SECAO  NR_TURNO                 NM_VOTAVEL  QT_VOTOS
0         102         1         2    FLAVIO NANTES BOLSONARO       207
1         102         1         2  LUIZ INÁCIO LULA DA SILVA        85
2         102         1         2                VOTO BRANCO         3
3         102         1         2                  VOTO NULO         2
4         102         2         2    FLAVIO NANTES BOLSONARO       185
...       ...       ...       ...                        ...       ...
1968      103       269         2                  VOTO NULO         8
1969      103       302         2    FLAVIO NANTES BOLSONARO        68
1970      103       302         2  LUIZ INÁCIO LULA DA SILVA       117
1971      103       302         2                VOTO BRANCO         5
1972      103       302         2                  VOTO NULO         4

[1973 rows x 5 columns]
Candidatos mapeados 1° turno (5 maiores totais de votos válidos):


,Votos
NM_VOTAVEL,
FLAVIO NANTES BOLSONARO,71478
LUIZ INÁCIO LULA DA SILVA,49403
AUGUSTO JORGE CURY,4568
VOTO NULO,4206
RENAN ANTONIO FERREIRA DOS SANTOS,3219


In [3]:
# Criando um CSV com as seções eleitorais e seus respectivos locais de votação e filtro de duplicados
colunas_secoes= ["NR_LOCAL_VOTACAO", "NM_LOCAL_VOTACAO", "DS_LOCAL_VOTACAO_ENDERECO"]
df_secoes = df[df["NM_MUNICIPIO"] == "DIVINÓPOLIS"][colunas_secoes]
df_secoes = df_secoes.drop_duplicates()
df_secoes["latitude"] = None
df_secoes["longitude"] = None
df_secoes.to_csv("locais_secoes_divinopolis.csv", index=False)

In [ ]:
# Definindo as coordenadas geográficas (latitude e longitude) para cada seção eleitoral em Divinópolis
import time
import re
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# 1. Configurar o geocodificador Nominatim (OpenStreetMap)
geolocator = Nominatim(user_agent="geocodificador_secoes_eleitorais_divinopolis")

# RateLimiter garante o intervalo de 1 segundo entre as requisições para evitar bloqueio
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.0)

def limpar_texto_rural(texto):
    """Remove termos de endereços rurais/genéricos que atrapalham a busca no Nominatim."""
    if pd.isna(texto):
        return ""
    
    texto = str(texto)
    # Lista de padrões e ruídos para remover
    padroes_para_remover = [
        r'\bs/?n\b', r'\bzona rural\b', r'\bpróximo\b.*', r'\bbarracão\b', 
        r'\bcomunidade\b', r'\bs/nº\b', r'\bsem número\b'
    ]
    for padrao in padroes_para_remover:
        texto = re.sub(padrao, '', texto, flags=re.IGNORECASE)
    
    # Remove espaços duplos e pontuações extras
    texto = re.sub(r'\s+', ' ', texto).strip(' ,.-')
    return texto

def buscar_coordenadas(row):
    # Se a latitude ou longitude já estiverem preenchidas, ignora e mantém os valores
    if pd.notna(row['latitude']) and pd.notna(row['longitude']):
        return row['latitude'], row['longitude']
    
    # Obtém e limpa o nome e endereço cadastrados
    nome_local = limpar_texto_rural(row['NM_LOCAL_VOTACAO'])
    endereco_local = limpar_texto_rural(row['DS_LOCAL_VOTACAO_ENDERECO'])
    
    # Monta a lista de tentativas ordenadas por especificidade (urbana e rural)
    tentativas = []
    
    # Tentativa 1: Nome do local + Endereço + Município
    if nome_local and endereco_local:
        tentativas.append(f"{nome_local}, {endereco_local}, Divinópolis, MG, Brasil")
    
    # Tentativa 2: Nome da escola/local + Município
    if nome_local:
        tentativas.append(f"{nome_local}, Divinópolis, MG, Brasil")
        
    # Tentativa 3: Endereço/Distrito isolado + Município (com fallback para povoados rurais)
    if endereco_local:
        tentativas.append(f"{endereco_local}, Divinópolis, MG, Brasil")
    
    # Executa a busca tentando cada formato até encontrar
    for query in tentativas:
        try:
            location = geocode(query)
            if location:
                print(f"[OK] Encontrado ('{query}'): -> ({location.latitude}, {location.longitude})")
                return location.latitude, location.longitude
        except Exception as e:
            print(f"[ERRO] Falha ao consultar '{query}': {e}")
            continue

    print(f"[X] Não encontrado: {row.get('NM_LOCAL_VOTACAO', 'Local sem nome')}")
    return None, None

def processar_csv(caminho_entrada, caminho_saida):
    print("Carregando arquivo CSV...")
    # Lê o CSV
    df = pd.read_csv(caminho_entrada, dtype=str, sep=',')

    print("Iniciando geocodificação dos locais...")
    
    # Aplica a busca linha por linha
    coordenadas = df.apply(buscar_coordenadas, axis=1)
    
    # Atualiza as colunas de latitude e longitude
    df['latitude'] = [coord[0] for coord in coordenadas]
    df['longitude'] = [coord[1] for coord in coordenadas]

    # Salva o resultado no arquivo de saída
    df.to_csv(caminho_saida, index=False, encoding='utf-8-sig')
    print(f"\nProcessamento concluído! Arquivo salvo em: {caminho_saida}")

# --- EXECUÇÃO DO SCRIPT ---
if __name__ == "__main__":
    ARQUIVO_ENTRADA = "locais_secoes_divinopolis_com_coordenadas.csv" # Originalmente o arquivo de entrada deve ser o "locais_secoes_divinopolis.csv" gerado anteriormente, mas aqui está definido como o arquivo de saída para evitar sobrescrever.
    ARQUIVO_SAIDA = "locais_secoes_divinopolis_com_coordenadas.csv"
    
    processar_csv(ARQUIVO_ENTRADA, ARQUIVO_SAIDA)

Carregando arquivo CSV...
Iniciando geocodificação dos locais...

Processamento concluído! Arquivo salvo em: locais_secoes_divinopolis_com_coordenadas.csv


In [ ]:
# Processamento para adicionar bairros e CEPs usando geocodificação reversa
import time
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable

def adicionar_bairros(caminho_entrada, caminho_saida):
    print("1. Carregando o arquivo CSV...")
    df = pd.read_csv(caminho_entrada)
    
    # 2. Configurar o Nominatim com timeout
    geolocator = Nominatim(user_agent="geocodificacao_reversa", timeout=10)
    
    # RateLimiter mantendo 1 segundo de intervalo mínimo entre chamadas normais
    reverse_geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1.0)
    
    bairros = []
    ceps = []
    
    print("2. Iniciando geocodificação reversa...")
    for idx, row in df.iterrows():
        lat = row['latitude']
        lon = row['longitude']
        
        bairro_encontrado = "Não identificado"
        cep_encontrado = "Não disponível"
        
        # Tentativas de reenvio caso o servidor caia ou dê timeout
        sucesso = False
        tentativas = 3
        
        for tentativa in range(1, tentativas + 1):
            try:
                # Faz a busca reversa
                location = reverse_geocode(f"{lat}, {lon}", language="pt")
                
                if location and 'address' in location.raw:
                    addr = location.raw['address']
                    bairro_encontrado = (
                        addr.get('suburb') or 
                        addr.get('neighbourhood') or 
                        addr.get('residential') or 
                        addr.get('city_district') or 
                        addr.get('quarter') or 
                        "Centro/Outro"
                    )
                    cep_encontrado = addr.get('postcode', "Não disponível")
                
                sucesso = True
                break # Sai do loop de tentativas se deu certo
                
            except (GeocoderTimedOut, GeocoderUnavailable) as e:
                print(f"[Aviso] Tentativa {tentativa}/{tentativas} falhou para a linha {idx+1} (Timeout/Servidor instável). Aguardando...")
                time.sleep(3 * tentativa) # Espera progressiva (3s, 6s, 9s) antes de tentar de novo
            except Exception as e:
                print(f"[ERRO] Erro inesperado na linha {idx+1}: {e}")
                break
        
        if sucesso:
            print(f"[OK] Linha {idx+1}: {row['NM_LOCAL_VOTACAO']} -> Bairro: {bairro_encontrado}")
        else:
            print(f"[X] Linha {idx+1}: Falhou após {tentativas} tentativas. Marcado como 'Não identificado'.")
            
        bairros.append(bairro_encontrado)
        ceps.append(cep_encontrado)
    
    # 3. Adiciona as novas colunas ao DataFrame
    df['bairro_osm'] = bairros
    df['cep_osm'] = ceps
    
    # 4. Salva o novo arquivo CSV
    df.to_csv(caminho_saida, index=False, encoding='utf-8-sig')
    print(f"\nProcessamento concluído! Arquivo salvo com sucesso em: {caminho_saida}")

# --- EXECUÇÃO DO SCRIPT ---
if __name__ == "__main__":
    ARQUIVO_ENTRADA = "locais_secoes_divinopolis_com_coordenadas.csv"
    ARQUIVO_SAIDA = "locais_secoes_divinopolis_com_bairros.csv"
    
    adicionar_bairros(ARQUIVO_ENTRADA, ARQUIVO_SAIDA)

1. Carregando o arquivo CSV...
2. Iniciando geocodificação reversa com proteção contra falhas de rede...
[OK] Linha 1: CENTRO PASTORAL DA IGREJA NOSSA SENHORA DA GUIA -> Bairro: Centro
[OK] Linha 2: ESCOLA ESTADUAL SÃO VICENTE -> Bairro: Niterói
[OK] Linha 3: ESCOLA MUNICIPAL CETEPE -> Bairro: Centro
[OK] Linha 4: ESCOLA ESTADUAL ILÍDIO DA COSTA PEREIRA -> Bairro: Alvorada
[OK] Linha 5: ESCOLA ESTADUAL ARMANDO NOGUEIRA SOARES -> Bairro: São José
[OK] Linha 6: ESCOLA MUNICIPAL ANTONIETA FONSECA -> Bairro: Quinta das Palmeiras
[OK] Linha 7: ESCOLA ESTADUAL ANTÔNIO DA COSTA PEREIRA -> Bairro: Santa Rosa
[OK] Linha 8: ESCOLA ESTADUAL HALIM SOUKI -> Bairro: Bela Vista
[OK] Linha 9: ESCOLA ESTADUAL ANTÔNIO BELARMINO GOMES -> Bairro: Florermida
[OK] Linha 10: ESCOLA ESTADUAL  MONSENHOR DOMINGOS -> Bairro: Santa Clara
[OK] Linha 11: ESCOLA ESTADUAL SÃO FRANCISCO DE ASSIS -> Bairro: Centro
[OK] Linha 12: ESCOLA ESTADUAL JOAQUIM NABUCO -> Bairro: Centro
[OK] Linha 13: ESCOLA MUNICIPAL ADOLFO MAC

# Visualização dos dados no 1° turno

### Interpretação dos dados
O mapa abaixo exibe o desempenho de cada candidato em Divinópolis no primeiro turno, onde os principais candidatos para a presidência tem sua própria cor e região de dominância na região da cidade baseado por seção eleitoral.
Em geral, é possível notar que o candidato Flávio Bolsonaro teve uma maior predominância de votos na cidade e nas zonas rurais, seguido de Lula da Silva principalmente em áreas mais periféricas e nos subúrbios da cidade nesse primeiro turno.

### Justificativa do design

Foi utilizado um preenchimento fill_opacity de 0.48 para permitir uma fácil visualização das áreas no mapa e orientações de voto em cada região da cidade enquanto permite a identificação clara do local/região, com as cores escolhidas para cada candidato sendo as mais comuns no meio político para candidatos de esquerda e direita (azul e vermelho) e cores diversificadas (verde escuro, amarelo, roxo, etc) para os demais candidatos a presidência.
A legenda descreve o nome de todos os candidatos que participaram do primeiro turno e a quantidade de seções em que ganharam na cidade, permitindo uma fácil comparação de quem obteve o maior número de votos e vitórias em área.

Para determinar as regiões e bairros da cidade para a criação do mapa cloroplético e estabelecer zonas de dominância de cada candidato, foram utilizados polígonos de Voronoi devido a dificuldade para obter definições precisas de bairros e áreas que seriam correspondentes a cada seção, dessa forma os polígonos estimam de forma aproximada uma área que abrange as seções incorporando toda a cidade e zonas rurais, recortadas para se manterem dentro dos limites do município graças a informações obtidas da API do IBGE, permitindo a geração deste tipo de mapa. Durante a passagem do primeiro para o segundo turno, algumas seções de votação tiveram seus locais de votação alterados, com o número de endereços distintos especificado ao gerar o mapa e sendo contornado com o uso do número, nome do local e o endereço que corresponde a cada turno para aquela seção para gerar informações nos locais onde elas se localizam.


In [8]:
from pathlib import Path
from html import escape
import json

import folium
import geopandas as gpd
import pandas as pd
from folium.plugins import MarkerCluster
from shapely.geometry import Polygon

CRS_METRICO = 'EPSG:31983'

arquivo_locais_mapa = Path('locais_secoes_divinopolis_com_bairros.csv')
if not arquivo_locais_mapa.exists():
    raise FileNotFoundError(f'Arquivo de locais e bairros não encontrado: {arquivo_locais_mapa.resolve()}')
# Limite municipal oficial do IBGE para Divinópolis (código 3122306).
arquivo_limite_municipal = Path('limite_municipal_divinopolis.geojson')
if not arquivo_limite_municipal.exists():
    raise FileNotFoundError(f'Limite municipal do IBGE não encontrado: {arquivo_limite_municipal.resolve()}')
limite_bruto = gpd.read_file(arquivo_limite_municipal)
if limite_bruto.empty or limite_bruto.crs is None:
    raise ValueError('O GeoJSON do limite municipal não contém geometria ou CRS definido.')
if 'codarea' not in limite_bruto.columns or set(limite_bruto['codarea'].astype(str)) != {'3122306'}:
    raise ValueError('O arquivo de limite não corresponde ao município de Divinópolis (IBGE 3122306).')
limite_municipal = limite_bruto.to_crs(CRS_METRICO).geometry.union_all()
if limite_municipal.is_empty or not limite_municipal.is_valid:
    raise ValueError('A geometria do limite municipal está vazia ou é inválida.')

colunas_resultados = {
    'ANO_ELEICAO', 'NR_TURNO', 'NR_ZONA', 'NR_SECAO', 'NR_LOCAL_VOTACAO',
    'NM_LOCAL_VOTACAO', 'DS_LOCAL_VOTACAO_ENDERECO',
    'DS_CARGO', 'NM_VOTAVEL', 'QT_VOTOS',
}
colunas_ausentes = colunas_resultados.difference(df_divinopolis.columns)
if colunas_ausentes:
    raise ValueError(f'Colunas ausentes nos resultados eleitorais: {sorted(colunas_ausentes)}')

resultados_presidente = df_divinopolis[
    df_divinopolis['DS_CARGO'].astype('string').str.strip().str.upper().eq('PRESIDENTE')
].copy()
resultados_presidente['ANO_ELEICAO'] = pd.to_numeric(resultados_presidente['ANO_ELEICAO'], errors='coerce')
anos_disponiveis = resultados_presidente['ANO_ELEICAO'].dropna()
if anos_disponiveis.empty:
    raise ValueError('Não há resultados presidenciais com ano de eleição válido.')
ano_eleicao = int(anos_disponiveis.max())

resultados_presidente['NR_TURNO'] = pd.to_numeric(resultados_presidente['NR_TURNO'], errors='coerce')
resultados_presidente['QT_VOTOS'] = pd.to_numeric(resultados_presidente['QT_VOTOS'], errors='coerce').fillna(0)
nomes_votaveis = resultados_presidente['NM_VOTAVEL'].astype('string').str.strip().str.upper()
resultados_presidente = resultados_presidente[
    (resultados_presidente['ANO_ELEICAO'] == ano_eleicao)
    & resultados_presidente['NR_TURNO'].isin([1, 2])
    & ~nomes_votaveis.isin(['VOTO BRANCO', 'VOTO NULO'])
    & resultados_presidente['NM_VOTAVEL'].notna()
].copy()
if resultados_presidente.empty:
    raise ValueError(f'Não há votos nominais para presidente em {ano_eleicao}.')

colunas_local = {
    'NR_LOCAL_VOTACAO', 'NM_LOCAL_VOTACAO', 'DS_LOCAL_VOTACAO_ENDERECO',
    'latitude', 'longitude', 'bairro_osm',
}
locais_mapa = pd.read_csv(arquivo_locais_mapa, encoding='utf-8-sig')
colunas_local_ausentes = colunas_local.difference(locais_mapa.columns)
if colunas_local_ausentes:
    raise ValueError(f'Colunas ausentes no arquivo de locais: {sorted(colunas_local_ausentes)}')

colunas_chave_local = ['NR_LOCAL_VOTACAO', '_nome_local_chave', '_endereco_local_chave']
for tabela in (locais_mapa, resultados_presidente):
    tabela['NR_LOCAL_VOTACAO'] = (
        tabela['NR_LOCAL_VOTACAO'].astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    )
    tabela['_nome_local_chave'] = (
        tabela['NM_LOCAL_VOTACAO'].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True).str.upper()
    )
    tabela['_endereco_local_chave'] = (
        tabela['DS_LOCAL_VOTACAO_ENDERECO'].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True).str.upper()
    )
    if tabela[colunas_chave_local].isna().any().any() or tabela[colunas_chave_local].eq('').any().any():
        raise ValueError('Há identificadores, nomes ou endereços de local vazios; não é possível associar as seções com segurança.')

if locais_mapa.duplicated(colunas_chave_local).any():
    raise ValueError('O arquivo de locais repete o mesmo NR_LOCAL_VOTACAO, nome e endereço.')
ids_locais_repetidos = locais_mapa.loc[
    locais_mapa['NR_LOCAL_VOTACAO'].duplicated(keep=False), 'NR_LOCAL_VOTACAO'
].nunique()
if ids_locais_repetidos:
    print(
        f'{ids_locais_repetidos} números de local aparecem em endereços distintos; '
        'as seções serão associadas pelo número, nome e endereço do turno correspondente.'
    )

locais_mapa['latitude'] = pd.to_numeric(locais_mapa['latitude'], errors='coerce')
locais_mapa['longitude'] = pd.to_numeric(locais_mapa['longitude'], errors='coerce')
locais_mapa['bairro_osm'] = (
    locais_mapa['bairro_osm'].astype('string').str.strip().replace('', pd.NA).fillna('Bairro não identificado')
)
locais_mapa = locais_mapa[
    locais_mapa['latitude'].between(-90, 90)
    & locais_mapa['longitude'].between(-180, 180)
].copy()
if locais_mapa.empty:
    raise ValueError('O arquivo de locais não contém coordenadas válidas.')

paleta = [
    '#225EA8', '#D73027', '#1A9850', '#FDAE61', '#756BB1', '#1D91C0',
    '#E7298A', '#66A61E', '#E6AB02', '#A6761D', '#666666', '#A6CEE3',
    '#FB9A99', '#B2DF8A', '#CAB2D6', '#FDBF6F',
]
candidatos = sorted(resultados_presidente['NM_VOTAVEL'].dropna().unique())
def normalizar_candidato(nome):
    import unicodedata
    return ''.join(
        caractere for caractere in unicodedata.normalize('NFKD', str(nome))
        if not unicodedata.combining(caractere)
    ).upper()


def cores_por_candidato(candidatos_turno):
    cores = {}
    cores_extras = iter(paleta[2:])
    for nome in candidatos_turno:
        chave = normalizar_candidato(nome)
        if 'FLAVIO' in chave and 'BOLSONARO' in chave:
            cores[nome] = '#1E5AA8'
        elif 'LUIZ INACIO LULA' in chave:
            cores[nome] = '#D71920'
        else:
            cores[nome] = next(cores_extras, '#506784')
    return cores


chaves_secao = ['NR_ZONA', 'NR_SECAO'] + colunas_chave_local


def formatar_votos(valor):
    return f'{int(valor):,}'.replace(',', '.')


def formatar_percentual(valor):
    return f'{valor:.1%}'.replace('.', ',')


def fecho_convexo(pontos):
    pontos_ordenados = sorted(set(pontos))
    if len(pontos_ordenados) < 3:
        raise ValueError('São necessários ao menos três locais de votação distintos para construir o Voronoi.')

    def produto_vetorial(origem, primeiro, segundo):
        return (
            (primeiro[0] - origem[0]) * (segundo[1] - origem[1])
            - (primeiro[1] - origem[1]) * (segundo[0] - origem[0])
        )

    inferior = []
    for ponto in pontos_ordenados:
        while len(inferior) >= 2 and produto_vetorial(inferior[-2], inferior[-1], ponto) <= 0:
            inferior.pop()
        inferior.append(ponto)

    superior = []
    for ponto in reversed(pontos_ordenados):
        while len(superior) >= 2 and produto_vetorial(superior[-2], superior[-1], ponto) <= 0:
            superior.pop()
        superior.append(ponto)

    contorno = inferior[:-1] + superior[:-1]
    if len(contorno) < 3:
        raise ValueError('Os locais de votação não formam uma área bidimensional para o Voronoi.')
    return contorno


def recortar_por_meio_plano(poligono, a, b, limite):
    if not poligono:
        return []

    resultado = []
    anterior = poligono[-1]
    valor_anterior = a * anterior[0] + b * anterior[1] - limite
    dentro_anterior = valor_anterior <= 1e-7

    for atual in poligono:
        valor_atual = a * atual[0] + b * atual[1] - limite
        dentro_atual = valor_atual <= 1e-7
        if dentro_atual != dentro_anterior:
            proporcao = valor_anterior / (valor_anterior - valor_atual)
            resultado.append((
                anterior[0] + proporcao * (atual[0] - anterior[0]),
                anterior[1] + proporcao * (atual[1] - anterior[1]),
            ))
        if dentro_atual:
            resultado.append(atual)
        anterior = atual
        valor_anterior = valor_atual
        dentro_anterior = dentro_atual

    return resultado


def poligonos_voronoi(pontos_metros, limites_municipio):
    pontos = [tuple(ponto) for ponto in pontos_metros]
    fecho_convexo(pontos)
    min_x, min_y, max_x, max_y = limites_municipio
    limite_externo = [
        (min_x, min_y),
        (max_x, min_y),
        (max_x, max_y),
        (min_x, max_y),
    ]
    celulas = []

    for indice, ponto in enumerate(pontos):
        celula = limite_externo.copy()
        for outro_indice, outro in enumerate(pontos):
            if indice == outro_indice:
                continue
            a = outro[0] - ponto[0]
            b = outro[1] - ponto[1]
            limite = (outro[0] ** 2 + outro[1] ** 2 - ponto[0] ** 2 - ponto[1] ** 2) / 2
            celula = recortar_por_meio_plano(celula, a, b, limite)
            if not celula:
                break
        if len(celula) >= 3:
            celulas.append(celula)
        else:
            raise ValueError('Não foi possível gerar uma célula de Voronoi para cada local de votação.')

    return celulas


def criar_mapa_turno(turno):
    resultados = resultados_presidente[resultados_presidente['NR_TURNO'] == turno]
    if resultados.empty:
        raise ValueError(f'Não há resultados presidenciais do {turno}º turno em {ano_eleicao}.')
    candidatos_turno = sorted(resultados['NM_VOTAVEL'].dropna().unique())
    if turno == 2 and len(candidatos_turno) != 2:
        raise ValueError(f'O segundo turno deveria ter dois candidatos, mas foram encontrados {len(candidatos_turno)}.')
    cores_turno = cores_por_candidato(candidatos_turno)

    totais = resultados.groupby(chaves_secao + ['NM_VOTAVEL'], as_index=False)['QT_VOTOS'].sum()
    votos_validos = (
        totais.groupby(chaves_secao, as_index=False)['QT_VOTOS'].sum()
        .rename(columns={'QT_VOTOS': 'votos_validos'})
    )
    vencedores_secao = (
        totais.sort_values(
            chaves_secao + ['QT_VOTOS', 'NM_VOTAVEL'],
            ascending=[True] * len(chaves_secao) + [False, True],
            kind='mergesort',
        )
        .drop_duplicates(chaves_secao)
        .rename(columns={'NM_VOTAVEL': 'lider_secao', 'QT_VOTOS': 'votos_lider_secao'})
    )
    secoes = votos_validos.merge(
        vencedores_secao[chaves_secao + ['lider_secao', 'votos_lider_secao']],
        on=chaves_secao,
        validate='one_to_one',
    )
    secoes['percentual_lider_secao'] = secoes['votos_lider_secao'] / secoes['votos_validos']

    dados_locais = totais.merge(
        locais_mapa[colunas_chave_local + ['latitude', 'longitude', 'NM_LOCAL_VOTACAO', 'bairro_osm']],
        on=colunas_chave_local,
        how='left',
        validate='many_to_one',
    )
    votos_por_local = (
        dados_locais.groupby(colunas_chave_local + ['latitude', 'longitude', 'NM_VOTAVEL'], as_index=False)
        ['QT_VOTOS'].sum()
    )
    vencedores_local = (
        votos_por_local.sort_values(
            colunas_chave_local + ['QT_VOTOS', 'NM_VOTAVEL'],
            ascending=[True] * len(colunas_chave_local) + [False, True],
            kind='mergesort',
        )
        .drop_duplicates(colunas_chave_local)
        .rename(columns={'NM_VOTAVEL': 'lider_local', 'QT_VOTOS': 'votos_lider_local'})
    )
    totais_local = (
        votos_por_local.groupby(colunas_chave_local, as_index=False)['QT_VOTOS'].sum()
        .rename(columns={'QT_VOTOS': 'votos_validos_local'})
    )
    vencedores_local = vencedores_local.merge(totais_local, on=colunas_chave_local, validate='one_to_one')
    vencedores_local['percentual_lider_local'] = (
        vencedores_local['votos_lider_local'] / vencedores_local['votos_validos_local']
    )

    secoes = secoes.merge(locais_mapa, on=colunas_chave_local, how='left', validate='many_to_one')
    secoes = secoes.merge(
        vencedores_local[colunas_chave_local + ['lider_local', 'votos_lider_local', 'votos_validos_local', 'percentual_lider_local']],
        on=colunas_chave_local,
        how='left',
        validate='many_to_one',
    )
    sem_coordenadas = secoes['latitude'].isna() | secoes['longitude'].isna()
    if sem_coordenadas.any():
        print(f'Seções sem coordenadas válidas, não exibidas: {int(sem_coordenadas.sum())}.')
    secoes = secoes[~sem_coordenadas].copy()
    if secoes.empty:
        raise ValueError(f'Nenhuma seção do {turno}º turno possui coordenadas válidas.')

    votos_por_ponto = (
        votos_por_local.groupby(['latitude', 'longitude', 'NM_VOTAVEL'], as_index=False)['QT_VOTOS'].sum()
    )
    vencedores_ponto = (
        votos_por_ponto.sort_values(
            ['latitude', 'longitude', 'QT_VOTOS', 'NM_VOTAVEL'],
            ascending=[True, True, False, True],
            kind='mergesort',
        )
        .drop_duplicates(['latitude', 'longitude'])
        .rename(columns={'NM_VOTAVEL': 'lider_ponto', 'QT_VOTOS': 'votos_lider_ponto'})
    )
    totais_ponto = (
        votos_por_ponto.groupby(['latitude', 'longitude'], as_index=False)['QT_VOTOS'].sum()
        .rename(columns={'QT_VOTOS': 'votos_validos_ponto'})
    )
    vencedores_ponto = vencedores_ponto.merge(
        totais_ponto, on=['latitude', 'longitude'], validate='one_to_one'
    )
    vencedores_ponto['percentual_lider_ponto'] = (
        vencedores_ponto['votos_lider_ponto'] / vencedores_ponto['votos_validos_ponto']
    )
    locais_por_ponto = (
        dados_locais.groupby(['latitude', 'longitude'], as_index=False)
        .agg(
            locais=('NM_LOCAL_VOTACAO', lambda nomes: ' / '.join(sorted(set(nomes.astype(str))))),
            bairro_osm=('bairro_osm', lambda bairros: ' / '.join(sorted(set(bairros.dropna().astype(str))))),
        )
    )
    vencedores_ponto = vencedores_ponto.merge(
        locais_por_ponto, on=['latitude', 'longitude'], validate='one_to_one'
    )

    latitude_centro = float(vencedores_ponto['latitude'].mean())
    longitude_centro = float(vencedores_ponto['longitude'].mean())
    pontos_geograficos = gpd.GeoSeries(
        gpd.points_from_xy(vencedores_ponto['longitude'], vencedores_ponto['latitude']),
        crs='EPSG:4326',
    )
    pontos_metros = [
        (ponto.x, ponto.y)
        for ponto in pontos_geograficos.to_crs(CRS_METRICO)
    ]
    celulas_voronoi = poligonos_voronoi(pontos_metros, limite_municipal.bounds)
    propriedades_voronoi = []
    for indice in range(len(celulas_voronoi)):
        registro = vencedores_ponto.iloc[indice]
        bairros = registro['bairro_osm'] or 'Não identificado'
        propriedades_voronoi.append({
            'local': registro['locais'],
            'bairros': bairros,
            'lider': registro['lider_ponto'],
            'votos': formatar_votos(registro['votos_lider_ponto']),
            'percentual': formatar_percentual(registro['percentual_lider_ponto']),
            'cor': cores_turno[registro['lider_ponto']],
        })
    areas_voronoi = gpd.GeoDataFrame(
        propriedades_voronoi,
        geometry=[Polygon(celula) for celula in celulas_voronoi],
        crs=CRS_METRICO,
    )
    areas_voronoi.geometry = areas_voronoi.geometry.intersection(limite_municipal)
    areas_voronoi = areas_voronoi[
        areas_voronoi.geometry.notna()
        & ~areas_voronoi.geometry.is_empty
        & areas_voronoi.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
    ]
    if areas_voronoi.empty:
        raise ValueError('Nenhuma área de Voronoi intersecta o limite municipal do IBGE.')
    features_voronoi = json.loads(areas_voronoi.to_crs('EPSG:4326').to_json())['features']
    limite_municipal_wgs84 = gpd.GeoSeries([limite_municipal], crs=CRS_METRICO).to_crs('EPSG:4326').iloc[0]

    mapa = folium.Map(
        location=[latitude_centro, longitude_centro],
        zoom_start=12,
        tiles='OpenStreetMap',
        control_scale=True,
    )
    folium.GeoJson(
        {'type': 'FeatureCollection', 'features': features_voronoi},
        name='Áreas aproximadas de Voronoi',
        style_function=lambda feature: {
            'fillColor': feature['properties']['cor'],
            'color': '#FFFFFF',
            'weight': 1.2,
            'fillOpacity': 0.48,
        },
        highlight_function=lambda feature: {'weight': 3, 'fillOpacity': 0.68},
        tooltip=folium.GeoJsonTooltip(
            fields=['local', 'bairros', 'lider', 'votos', 'percentual'],
            aliases=['Local(is) de votação:', 'Bairro(s) do local (referência):', 'Candidato líder:', 'Votos no local:', 'Percentual dos válidos:'],
            localize=True,
            sticky=True,
            labels=True,
            class_name='votacao-tooltip',
        ),
    ).add_to(mapa)
    folium.GeoJson(
        limite_municipal_wgs84.__geo_interface__,
        name='Limite municipal oficial (IBGE)',
        style_function=lambda feature: {
            'color': '#202124',
            'weight': 2.5,
            'fillOpacity': 0,
        },
    ).add_to(mapa)
    min_longitude, min_latitude, max_longitude, max_latitude = limite_municipal_wgs84.bounds
    mapa.fit_bounds(
        [
            [min_latitude, min_longitude],
            [max_latitude, max_longitude],
        ],
        padding=[24, 24],
    )
    agrupamento = MarkerCluster(
        name='Seções eleitorais', show_coverage_on_hover=False, spiderfy_on_max_zoom=True
    ).add_to(mapa)
    votos_por_secao = (
        totais.pivot(index=chaves_secao, columns='NM_VOTAVEL', values='QT_VOTOS')
        .reindex(columns=candidatos_turno, fill_value=0)
        .fillna(0)
    )

    for _, secao in secoes.iterrows():
        chave = tuple(secao[coluna] for coluna in chaves_secao)
        votos_secao = votos_por_secao.loc[chave]
        linhas_votos = ''.join(
            '<tr><td>{}</td><td>{}</td><td>{}</td></tr>'.format(
                escape(str(candidato)),
                formatar_votos(votos_secao[candidato]),
                formatar_percentual(votos_secao[candidato] / secao['votos_validos']),
            )
            for candidato in sorted(candidatos_turno, key=lambda nome: (-votos_secao[nome], nome))
        )
        cor = cores_turno[secao['lider_secao']]
        tooltip_html = (
            '<div class="votacao-card">'
            f'<div class="votacao-bairro">{escape(str(secao["bairro_osm"]))}</div>'
            f'<div class="votacao-local">{escape(str(secao["NM_LOCAL_VOTACAO"]))}</div>'
            f'<div class="votacao-secao">Zona {int(secao["NR_ZONA"])} · Seção {int(secao["NR_SECAO"])}</div>'
            f'<div class="votacao-lider">Líder na seção: <strong>{escape(str(secao["lider_secao"]))}</strong> '
            f'({formatar_percentual(secao["percentual_lider_secao"])}; '
            f'{formatar_votos(secao["votos_lider_secao"])} votos)</div>'
            f'<div class="votacao-regiao">Líder no local: {escape(str(secao["lider_local"]))} '
            f'({formatar_percentual(secao["percentual_lider_local"])})</div>'
            '<table><thead><tr><th>Candidato</th><th>Votos</th><th>% válidos</th></tr></thead>'
            f'<tbody>{linhas_votos}</tbody></table>'
            '</div>'
        )
        folium.Marker(
            location=[secao['latitude'], secao['longitude']],
            icon=folium.DivIcon(
                icon_size=(16, 16), icon_anchor=(8, 8),
                html=f'<div style="width:14px;height:14px;border:2px solid #fff;border-radius:50%;background:{cor};box-shadow:0 1px 4px #333;"></div>',
            ),
            tooltip=folium.Tooltip(
                tooltip_html, sticky=True, opacity=0.97, class_name='votacao-tooltip'
            ),
        ).add_to(agrupamento)

    contagem_areas = vencedores_ponto['lider_ponto'].value_counts().to_dict()
    itens_legenda = ''.join(
        f'<div class="legenda-item"><span class="legenda-cor" style="background:{cores_turno[candidato]}"></span>'
        f'<span>{escape(str(candidato))}</span><b>{int(contagem_areas.get(candidato, 0))}</b></div>'
        for candidato in candidatos_turno
    )
    estilo = """
    <style>
      .votacao-tooltip { background: #fff; border: 0; border-radius: 6px; box-shadow: 0 2px 10px rgba(0,0,0,.24); padding: 8px 10px; font: 12px/1.4 Arial,sans-serif; }
      .votacao-tooltip:before { display: none; }
      .votacao-card { min-width: 290px; max-width: 390px; max-height: 320px; overflow-y: auto; padding: 10px 12px; border-radius: 8px; background: #fff; color: #202124; box-shadow: 0 2px 12px rgba(0,0,0,.28); font: 12px/1.4 Arial,sans-serif; }
      .votacao-bairro { font-size: 15px; font-weight: 700; }
      .votacao-local { color: #555; margin: 2px 0; }
      .votacao-secao { font-weight: 700; margin: 6px 0; }
      .votacao-lider { margin: 5px 0; }
      .votacao-card table { width: 100%; border-collapse: collapse; margin-top: 6px; }
      .votacao-card th, .votacao-card td { padding: 3px 4px; border-bottom: 1px solid #e5e7eb; text-align: left; }
      .votacao-card th:nth-child(n+2), .votacao-card td:nth-child(n+2) { text-align: right; }
      .votacao-regiao { margin-top: 6px; color: #555; font-size: 11px; }
      .mapa-legenda { position: fixed; z-index: 9999; top: 12px; right: 12px; max-width: 310px; max-height: 45vh; overflow-y: auto; padding: 12px; border-radius: 8px; background: rgba(255,255,255,.96); box-shadow: 0 1px 8px rgba(0,0,0,.3); font: 12px/1.4 Arial,sans-serif; }
      .mapa-legenda h4 { margin: 0 0 8px; font-size: 14px; }
      .legenda-item { display: grid; grid-template-columns: 14px 1fr auto; gap: 6px; align-items: center; margin: 4px 0; }
      .legenda-cor { width: 12px; height: 12px; border-radius: 50%; border: 1px solid #777; }
      .legenda-nota { margin-top: 8px; color: #555; }
    </style>
    """
    legenda = (
        f'<div class="mapa-legenda"><h4>Presidente · {turno}º turno · {ano_eleicao}</h4>'
        f'{itens_legenda}<div class="legenda-nota">Cada polígono representa a área aproximada mais próxima '
        'de um local de votação; a cor indica o candidato com mais votos somados nas seções desse local. '
        'As células de Voronoi foram recortadas ao limite municipal oficial do IBGE; não representam limites de bairros. '
        'Os pontos mostram as seções eleitorais; clique nos agrupamentos para ampliá-los.</div></div>'
    )
    mapa.get_root().html.add_child(folium.Element(estilo + legenda))
    folium.LayerControl(collapsed=True).add_to(mapa)
    print(f'Mapa presidencial do {turno}º turno de {ano_eleicao}: {len(secoes)} seções e {len(features_voronoi)} áreas aproximadas de Voronoi.')
    return mapa

mapa_1turno = criar_mapa_turno(1)
display(mapa_1turno)


26 números de local aparecem em endereços distintos; as seções serão associadas pelo número, nome e endereço do turno correspondente.
Mapa presidencial do 1º turno de 2026: 501 seções e 73 áreas aproximadas de Voronoi.


# Visualização dos dados no 2° turno

### Interpretação dos dados
O mapa abaixo exibe o desempenho de cada candidato em Divinópolis no segundo turno, onde os 2 candidatos para a presidência, sendo Flávio Bolsonaro e Lula da Silva, sendo divididos pela mesma cor do primeiro turno (azul e vermelho) e sendo exibidos por dominância na região da cidade baseado por seção eleitoral.
Em geral, é possível notar que o candidato Flávio Bolsonaro ainda teve um maior número de votos na cidade no segundo turno, mas com um aumento nas áreas de dominância e nos números de Lula da Silva nesse segundo turno com a conquista de novas regiões que no primeiro turno estavam com a maioria dos votos para Flávio, um fato curioso e que pela análise de números indica que em algumas seções eleitorais rurais o número de votos para Flávio continuaram iguais aos do primeiro turno mas no segundo turno os números de votos para o Lula aumentaram na mesma seção comparado ao primeiro e culminaram na sua vitória na área, com a possível causa sendo a redução no número de abstenções e também a passagem do voto que antes eram de outros candidatos para ele.

### Justificativa do design

Foi utilizado um preenchimento fill_opacity de 0.48 para permitir uma fácil visualização das marcações no mapa e orientações de voto em cada região da cidade igual ao utilizado no mapa do primeiro turno, com as cores escolhidas para cada candidato sendo as mesmas utilizadas no mapa do primeiro turno para manter o padrão e por serem indicativos clássicos dos partidos de cada um.
As características do mapa, polígonos e informações continuaram iguais, com a principal modificação sendo a legenda que agora indica apenas os dois candidatos que participaram do segundo turno e o número de seções em que venceram na cidade.

In [9]:
from IPython.display import display
from pathlib import Path

mapa_2turno = criar_mapa_turno(2)
display(mapa_2turno)

diretorio_exportacao = Path('mapas_web')
diretorio_exportacao.mkdir(parents=True, exist_ok=True)
arquivo_mapa_1turno = diretorio_exportacao / 'mapa_presidencial_1_turno.html'
arquivo_mapa_2turno = diretorio_exportacao / 'mapa_presidencial_2_turno.html'
arquivo_indice = diretorio_exportacao / 'index.html'

mapa_1turno.save(str(arquivo_mapa_1turno))
mapa_2turno.save(str(arquivo_mapa_2turno))
arquivo_indice.write_text(
    '''<!doctype html>
<html lang="pt-BR">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Mapas eleitorais de Divinópolis</title>
  <style>
    body { max-width: 720px; margin: 3rem auto; padding: 0 1rem; font: 16px/1.5 Arial, sans-serif; color: #202124; }
    li { margin: .8rem 0; }
  </style>
</head>
<body>
  <h1>Resultados presidenciais de Divinópolis</h1>
  <ul>
    <li><a href="mapa_presidencial_1_turno.html">Mapa do 1º turno</a></li>
    <li><a href="mapa_presidencial_2_turno.html">Mapa do 2º turno</a></li>
  </ul>
  <p>As áreas de Voronoi são recortadas ao limite municipal oficial do IBGE e não representam limites de bairros. A visualização requer conexão com a internet para carregar o mapa-base e os recursos do Leaflet.</p>
</body>
</html>
''',
    encoding='utf-8',
)
print(f'Mapas HTML e índice salvos em: {diretorio_exportacao.resolve()}')

Mapa presidencial do 2º turno de 2026: 501 seções e 73 áreas aproximadas de Voronoi.


Mapas HTML e índice salvos em: D:\1-USUÁRIO\Downloads\Atividade_Folium\mapas_web
